In [ ]:
import os
import sys
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from collections import defaultdict
from contextlib import nullcontext
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

# 1. CONFIGURATION


In [ ]:
class Config:
    seed = 42
    model_name = "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"
    img_size = 448

    batch_size = 10
    samples_per_class = 2
    grad_accum = 3
    
    # LP-FT Epochs
    num_epochs_lp = 5    # Linear Probing (Head Only)
    num_epochs_ft = 8    # Fine-Tuning (Whole Model)
    
    # Learning Rates
    lr_lp = 1e-3         # Fast LR to align the random head
    lr_ft_backbone = 2e-6 # TINY LR to prevent catastrophic forgetting
    lr_ft_head = 1e-4    

    weight_decay = 1e-3
    arcface_s = 30.0
    arcface_m = 0.50
    triplet_margin = 0.3
    triplet_weight = 0.3

    use_tta = True
    use_qe = True
    qe_top_k = 3
    use_rerank = True
    rerank_k1 = 20
    rerank_k2 = 6
    rerank_lambda = 0.2
    blend_raw_ratio = 0.20

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device_type = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = (device_type == "cuda")

    augmented_csv = "/kaggle/input/datasets/sarthaksachan/new-augmentation/augmented_train_ouroboros.csv" # Ensure this is from the 0.948 CSV
    test_csv = "/kaggle/input/competitions/jaguar-re-id/test.csv"
    test_dir = "/kaggle/input/competitions/jaguar-re-id/test/test"
    pretrained_path = "/kaggle/input/datasets/sanidhyavijay24/jaguar-eva-model1/model.pth" 

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(Config.seed)

# 2. DATASET & TRANSFORMS

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((Config.img_size, Config.img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=15, translate=(0.15, 0.15), scale=(0.85, 1.15)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.481, 0.457, 0.408], [0.268, 0.261, 0.275]),
    transforms.RandomErasing(p=0.25),
])

test_transform = transforms.Compose([
    transforms.Resize((Config.img_size, Config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.481, 0.457, 0.408], [0.268, 0.261, 0.275]),
])

class AugmentedJaguarDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False, img_dir=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
        self.img_dir = Path(img_dir) if img_dir else None

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / row["filename"] if self.is_test else row["filepath"]
        try: img = Image.open(img_path).convert("RGB")
        except Exception: img = Image.new("RGB", (Config.img_size, Config.img_size))
        if self.transform: img = self.transform(img)
        if self.is_test: return img, row["filename"]
        return img, torch.tensor(int(row["label"]), dtype=torch.long)

class BalancedPKSampler:
    def __init__(self, labels, batch_size, samples_per_class=2):
        self.labels = [int(x) for x in labels]
        self.batch_size = int(batch_size)
        self.k = int(samples_per_class)
        self.p = self.batch_size // self.k
        self.class_indices = defaultdict(list)
        for idx, lab in enumerate(self.labels): self.class_indices[lab].append(idx)
        self.classes = list(self.class_indices.keys())

    def __len__(self): return len(self.labels) // self.batch_size
    def __iter__(self):
        for _ in range(len(self)):
            batch = []
            selected_classes = random.sample(self.classes, k=self.p) if len(self.classes) >= self.p else random.choices(self.classes, k=self.p)
            for cls in selected_classes:
                idxs = self.class_indices[cls]
                chosen = random.sample(idxs, k=self.k) if len(idxs) >= self.k else random.choices(idxs, k=self.k)
                batch.extend(chosen)
            yield batch


# 3. MODEL ARCHITECTURE

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        x = x.clamp(min=self.eps).pow(self.p)
        x = F.avg_pool2d(x, kernel_size=(x.size(-2), x.size(-1)))
        return x.pow(1.0 / self.p)

class ArcFaceLayer(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5):
        super().__init__()
        self.s, self.m = float(s), float(m)
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, emb, labels=None):
        cosine = F.linear(F.normalize(emb, dim=1), F.normalize(self.weight, dim=1)).clamp(-1.0, 1.0)
        if labels is None: return cosine * self.s
        sine = torch.sqrt(torch.clamp(1.0 - cosine * cosine, min=0.0))
        phi = cosine * math.cos(self.m) - sine * math.sin(self.m)
        phi = torch.where(cosine > math.cos(math.pi - self.m), phi, cosine - (math.sin(math.pi - self.m) * self.m))
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        return ((one_hot * phi) + ((1.0 - one_hot) * cosine)) * self.s

class BatchHardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)
    def forward(self, embeddings, labels):
        dist_mat = torch.cdist(embeddings, embeddings, p=2)
        labels = labels.view(-1, 1)
        eye = torch.eye(dist_mat.size(0), device=dist_mat.device, dtype=torch.bool)
        is_pos = labels.eq(labels.t()) & (~eye)
        is_neg = labels.ne(labels.t())
        dist_ap = dist_mat.masked_fill(~is_pos, float("-inf")).max(dim=1).values
        dist_an = dist_mat.masked_fill(~is_neg, float("inf")).min(dim=1).values
        dist_ap = torch.where(torch.isfinite(dist_ap), dist_ap, torch.zeros_like(dist_ap))
        dist_an = torch.where(torch.isfinite(dist_an), dist_an, torch.zeros_like(dist_an))
        return self.ranking_loss(dist_an, dist_ap, torch.ones_like(dist_an))

class EVABoss(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model(Config.model_name, pretrained=True, num_classes=0)
        if hasattr(self.backbone, "set_grad_checkpointing"): self.backbone.set_grad_checkpointing(True)
        self.feat_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim")
        self.gem = GeM(p=3)
        self.bn = nn.BatchNorm1d(self.feat_dim)
        self.head = ArcFaceLayer(self.feat_dim, num_classes, s=Config.arcface_s, m=Config.arcface_m)

    def _to_feature_map(self, features: torch.Tensor) -> torch.Tensor:
        if features.dim() == 4: return features
        B, N, C = features.shape
        if hasattr(self.backbone, "patch_embed") and hasattr(self.backbone.patch_embed, "num_patches"):
            num_patches = int(self.backbone.patch_embed.num_patches)
            if N == num_patches + 1:
                features = features[:, 1:, :]
                N = features.size(1)
        else:
            rN, rNm1 = int(math.sqrt(N)), int(math.sqrt(max(N - 1, 1)))
            if rN * rN != N and rNm1 * rNm1 == (N - 1):
                features = features[:, 1:, :]
                N = features.size(1)
        H = W = int(math.sqrt(N))
        if H * W != N:
            sq = int(math.sqrt(N)) ** 2
            features = features[:, :sq, :]
            N = features.size(1)
            H = W = int(math.sqrt(N))
        return features.transpose(1, 2).contiguous().reshape(B, C, H, W)

    def forward(self, x, labels=None):
        features = self.backbone.forward_features(x)
        if isinstance(features, dict): features = features.get("x", None) or features.get("last_hidden_state", None) or next(iter(features.values()))
        if isinstance(features, (tuple, list)): features = features[0]
        fmap = self._to_feature_map(features)
        emb = self.bn(self.gem(fmap).flatten(1))
        normed = F.normalize(emb, dim=1)
        if labels is not None: return self.head(emb, labels), normed
        return normed

# 4. POST-PROCESSING (WITH EXPLOIT & MS-TTA)

In [ ]:
@torch.no_grad()
def extract_features_mstta(model, loader):
    model.eval()
    all_features, all_names = [], []
    autocast_ctx = torch.amp.autocast(device_type=Config.device_type, enabled=Config.use_amp) \
        if hasattr(torch, "amp") else (torch.cuda.amp.autocast(enabled=Config.use_amp) if Config.use_amp else nullcontext())

    pbar = tqdm(loader, desc="Extracting MS-TTA", file=sys.stdout, ncols=100, mininterval=2.0)
    for imgs, img_names in pbar:
        imgs = imgs.to(Config.device, non_blocking=True)
        
        with autocast_ctx:
            f1 = model(imgs)
            f2 = model(torch.flip(imgs, [3]))
            
            upscale_size = 512
            imgs_512 = F.interpolate(imgs, size=(upscale_size, upscale_size), mode='bilinear', align_corners=False)
            y1 = (upscale_size - Config.img_size) // 2
            x1 = (upscale_size - Config.img_size) // 2
            imgs_zoom = imgs_512[:, :, y1:y1+Config.img_size, x1:x1+Config.img_size]
            
            f3 = model(imgs_zoom)
            f4 = model(torch.flip(imgs_zoom, [3])) 
            
            f_avg = (f1 + f2 + f3 + f4) / 4.0
            
        all_features.append(F.normalize(f_avg, dim=1).cpu())
        all_names.extend(img_names)
        
    return torch.cat(all_features, dim=0).numpy(), all_names

def query_expansion(embeddings, top_k=3):
    print("Applying Query Expansion...")
    sims = embeddings @ embeddings.T
    indices = np.argsort(-sims, axis=1)[:, :top_k]
    new_emb = np.zeros_like(embeddings)
    for i in range(len(embeddings)):
        new_emb[i] = np.mean(embeddings[indices[i]], axis=0)
    return new_emb / (np.linalg.norm(new_emb, axis=1, keepdims=True) + 1e-12)

def k_reciprocal_rerank(similarity_matrix, k1=20, k2=6, lambda_value=0.2):
    print(f"Applying Re-ranking (λ={lambda_value})...")
    distance_matrix = 1 - similarity_matrix
    original_dist = distance_matrix.copy()
    initial_rank = np.argsort(distance_matrix, axis=1)

    nn_k1 = []
    for i in range(len(similarity_matrix)):
        forward_k1 = initial_rank[i, :k1 + 1]
        backward_k1 = initial_rank[forward_k1, :k1 + 1]
        fi = np.where(backward_k1 == i)[0]
        nn_k1.append(forward_k1[fi])

    jaccard_dist = np.zeros_like(distance_matrix) 
    
    for i in range(len(similarity_matrix)):
        ind_non_zero = np.where(distance_matrix[i, :] < 0.6)[0]
        ind_images = [j for j in ind_non_zero if len(np.intersect1d(nn_k1[i], nn_k1[j])) > 0]
        for j in ind_images:
            inter = len(np.intersect1d(nn_k1[i], nn_k1[j]))
            uni = len(np.union1d(nn_k1[i], nn_k1[j]))
            if uni > 0:
                jaccard_dist[i, j] = 1 - inter / uni

    final_dist = jaccard_dist * lambda_value + original_dist * (1 - lambda_value)
    return 1 - final_dist

# 5. MASTER LP-FT EXECUTION LOOP

In [ ]:
def run_lp_ft_pipeline():
    print("=" * 80)
    print("LP-FT OUROBOROS: SOLVING CATASTROPHIC FORGETTING")
    print("=" * 80)

    df = pd.read_csv(Config.augmented_csv)
    num_classes = int(df["label"].nunique())

    train_dataset = AugmentedJaguarDataset(df, transform=train_transform)
    sampler = BalancedPKSampler(df["label"], Config.batch_size, Config.samples_per_class)
    train_loader = DataLoader(
        train_dataset, batch_sampler=sampler, num_workers=2, 
        pin_memory=(Config.device_type == "cuda"), persistent_workers=True
    )

    model = EVABoss(num_classes=num_classes).to(Config.device)

    print(f"Loading 0.938 Foundation from {Config.pretrained_path}...")
    ckpt = torch.load(Config.pretrained_path, map_location=Config.device)
    state_dict = ckpt.get("model_state_dict", ckpt.get("model", ckpt))
    state_dict = {k: v for k, v in state_dict.items() if "head" not in k and "arcface" not in k}
    model.load_state_dict(state_dict, strict=False)

    criterion_arc = nn.CrossEntropyLoss()
    criterion_tri = BatchHardTripletLoss(margin=Config.triplet_margin)

    autocast_ctx = torch.amp.autocast(device_type=Config.device_type, enabled=Config.use_amp) if hasattr(torch, "amp") else (torch.cuda.amp.autocast(enabled=Config.use_amp) if Config.use_amp else nullcontext())
    scaler = torch.amp.GradScaler(device=Config.device_type, enabled=Config.use_amp) if hasattr(torch, "amp") else (torch.cuda.amp.GradScaler(enabled=Config.use_amp) if Config.use_amp else None)

    # ==========================================
    # PHASE A: LINEAR PROBING (FREEZE BACKBONE)
    # ==========================================
    print("PHASE A: LINEAR PROBING (Aligning Head)")
    
    for param in model.backbone.parameters(): param.requires_grad = False
    for param in model.gem.parameters(): param.requires_grad = False
    for param in model.bn.parameters(): param.requires_grad = False

    optimizer_lp = torch.optim.AdamW(model.head.parameters(), lr=Config.lr_lp, weight_decay=Config.weight_decay)
    scheduler_lp = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_lp, T_max=Config.num_epochs_lp)

    for epoch in range(Config.num_epochs_lp):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"LP Epoch {epoch+1}/{Config.num_epochs_lp}", leave=True, file=sys.stdout, ncols=100, mininterval=5.0)
        optimizer_lp.zero_grad(set_to_none=True)

        for step, (imgs, labs) in enumerate(pbar):
            imgs, labs = imgs.to(Config.device, non_blocking=True), labs.to(Config.device, non_blocking=True)
            with autocast_ctx:
                logits, embeddings = model(imgs, labs)
                loss = criterion_arc(logits, labs) / Config.grad_accum

            if Config.use_amp: scaler.scale(loss).backward()
            else: loss.backward()

            if ((step + 1) % Config.grad_accum == 0) or ((step + 1) == len(train_loader)):
                if Config.use_amp:
                    scaler.step(optimizer_lp)
                    scaler.update()
                else: optimizer_lp.step()
                optimizer_lp.zero_grad(set_to_none=True)

            running_loss += loss.item() * Config.grad_accum
            pbar.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        scheduler_lp.step()
        print(f"LP Epoch {epoch+1} Loss: {running_loss / max(1, len(train_loader)):.4f}")

    # ==========================================
    # PHASE B: FINE-TUNING (UNFREEZE ALL)
    # ==========================================
    print("PHASE B: FINE-TUNING (Triplet Optimization)")
    
    for param in model.parameters(): param.requires_grad = True

    optimizer_ft = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': Config.lr_ft_backbone},
        {'params': model.gem.parameters(), 'lr': Config.lr_ft_backbone},
        {'params': model.bn.parameters(), 'lr': Config.lr_ft_backbone},
        {'params': model.head.parameters(), 'lr': Config.lr_ft_head} 
    ], weight_decay=Config.weight_decay)
    
    scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=Config.num_epochs_ft)
    best_epoch_loss = float("inf")

    for epoch in range(Config.num_epochs_ft):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"FT Epoch {epoch+1}/{Config.num_epochs_ft}", leave=True, file=sys.stdout, ncols=100, mininterval=5.0)
        optimizer_ft.zero_grad(set_to_none=True)

        for step, (imgs, labs) in enumerate(pbar):
            imgs, labs = imgs.to(Config.device, non_blocking=True), labs.to(Config.device, non_blocking=True)
            with autocast_ctx:
                logits, embeddings = model(imgs, labs)
                loss = (criterion_arc(logits, labs) + Config.triplet_weight * criterion_tri(embeddings, labs)) / Config.grad_accum

            if Config.use_amp: scaler.scale(loss).backward()
            else: loss.backward()

            if ((step + 1) % Config.grad_accum == 0) or ((step + 1) == len(train_loader)):
                if Config.use_amp:
                    scaler.step(optimizer_ft)
                    scaler.update()
                else: optimizer_ft.step()
                optimizer_ft.zero_grad(set_to_none=True)

            running_loss += loss.item() * Config.grad_accum
            pbar.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        scheduler_ft.step()
        epoch_loss = running_loss / max(1, len(train_loader))
        print(f" FT Epoch {epoch+1} Loss: {epoch_loss:.4f}")

        if epoch_loss < best_epoch_loss - 1e-8:
            best_epoch_loss = float(epoch_loss)
            torch.save({
                "model_state_dict": model.state_dict(),
                "num_classes": num_classes
            }, "eva02_ouroboros_final.pth")
            print(f" New Best LP-FT Model Saved!")
            # ==========================================
            # FINAL MS-TTA INFERENCE
            # ==========================================
            print("\n" + "=" * 80)
            print(" MS-TTA INFERENCE ON OUROBOROS MODEL")
            print("=" * 80)
            
            model.load_state_dict(torch.load("eva02_ouroboros_final.pth", map_location=Config.device)["model_state_dict"])
            
            test_df = pd.read_csv(Config.test_csv)
            unique_test = sorted(list(set(test_df["query_image"]) | set(test_df["gallery_image"])))
            
            test_ds = AugmentedJaguarDataset(pd.DataFrame({"filename": unique_test}), transform=test_transform, is_test=True, img_dir=Config.test_dir)
            test_loader = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=2)
        
            embeddings, names = extract_features_mstta(model, test_loader)
            img_map = {n: i for i, n in enumerate(names)}
        
            print("Computing raw cosine similarity...")
            sim_raw = embeddings @ embeddings.T
        
            if Config.use_qe:
                embeddings_qe = query_expansion(embeddings, top_k=Config.qe_top_k)
                sim_qe = embeddings_qe @ embeddings_qe.T
            else: sim_qe = sim_raw
        
            if Config.use_rerank:
                sim_rerank = k_reciprocal_rerank(sim_qe, k1=Config.rerank_k1, k2=Config.rerank_k2, lambda_value=Config.rerank_lambda)
            else: sim_rerank = sim_qe
        
            print(f"\n Blending: {Config.blend_raw_ratio:.0%} raw + {1-Config.blend_raw_ratio:.0%} reranked")
            final_sim_matrix = (Config.blend_raw_ratio * sim_raw) + ((1 - Config.blend_raw_ratio) * sim_rerank)
        
            q_idx = test_df["query_image"].map(img_map).to_numpy()
            g_idx = test_df["gallery_image"].map(img_map).to_numpy()
            preds = np.clip(final_sim_matrix[q_idx, g_idx], 0.0, 1.0)
        
            sub = pd.DataFrame({"row_id": test_df["row_id"], "similarity": preds})
            sub.to_csv("submission_ouroboros_final.csv", index=False)
            print(f"\n FINAL Ouroboros Submission Saved! Mean Sim: {float(np.mean(preds)):.4f}")

In [ ]:
if __name__ == "__main__":
    run_lp_ft_pipeline()